# Movie Revenue Correlation Analysis

This notebook investigates which movie features are most associated with gross revenue. The story is simple: start with raw movie data, check quality, preserve the original dataset, analyze numeric relationships, then test whether encoded categorical fields add useful signal.

The main lesson is technical and analytical: correlation can help find patterns, but it does not prove causation.

## 1. Business question

Which available movie attributes appear most related to gross revenue?

We will focus on:

- Budget
- Votes
- Score
- Runtime
- Year
- Company
- Genre

The target metric is `gross`.

## 2. Load libraries and data

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Show all columns
pd.set_option('display.max_columns', None)

# Load source data
df = pd.read_csv('../data/movies.csv')

df.head()

## 3. Inspect the dataset

Before looking for correlations, we need to understand the shape, columns, data types, and missing values.

In [ ]:
# Dataset size
df.shape

In [ ]:
# Column names
df.columns

In [ ]:
# Data types
df.info()

In [ ]:
# Missing values
df.isna().sum().sort_values(ascending=False)

## 4. Prepare fields safely

The important fix is to preserve `df` as the full dataset. We should not replace it with only numeric columns, because later we need text fields like `company`.

We clean column names and convert known numeric columns, but we keep all original columns in `df`.

In [ ]:
# Clean column names
df.columns = df.columns.str.strip().str.lower()

# Define numeric fields
numeric_columns = ['year', 'score', 'votes', 'budget', 'gross', 'runtime']

# Convert numeric fields
for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors='coerce')

df.head()

In [ ]:
# Confirm preserved columns
df.columns

## 5. Review gross revenue

`gross` is the revenue field we want to explain. It has very large values and visible outliers, which is normal for movie revenue because a small number of movies earn much more than the rest.

In [ ]:
# Revenue summary
df['gross'].describe()

In [ ]:
# Check revenue outliers
plt.figure(figsize=(8, 4))
df.boxplot(column=['gross'])
plt.title('Gross Revenue Outliers')
plt.ylabel('Gross Revenue')
plt.show()

## 6. First relationship: score vs gross

This chart checks whether higher-rated movies tend to earn more. A regression line helps show the overall direction, but the scatter shows that score alone is not enough to explain revenue.

In [ ]:
# Plot score relationship
plt.figure(figsize=(10, 6))

sns.regplot(
    x='score',
    y='gross',
    data=df,
    scatter_kws={'alpha': 0.45, 's': 35},
    line_kws={'color': 'red', 'linewidth': 2}
)

plt.title('Movie Score vs Gross Revenue')
plt.xlabel('IMDb Score')
plt.ylabel('Gross Revenue')
plt.grid(alpha=0.2)
plt.show()

## 7. Correlation between numeric features

For numeric correlation, we create `numeric_df`. This avoids the earlier mistake of overwriting `df` and losing columns like `company`.

In [ ]:
# Keep numeric copy
numeric_df = df.select_dtypes(include=['number'])

# Pearson correlation
correlation_matrix = numeric_df.corr(method='pearson')

correlation_matrix

In [ ]:
# Plot numeric correlations
plt.figure(figsize=(10, 6))

sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap='coolwarm',
    linewidths=0.5,
    fmt='.2f'
)

plt.title('Correlation Matrix for Numeric Features')
plt.xlabel('Movie Features')
plt.ylabel('Movie Features')
plt.show()

## 8. Rank correlations with gross

The heatmap is useful, but a ranked table makes the story clearer. We sort by absolute correlation with `gross`.

In [ ]:
# Rank gross correlations
gross_correlations = (
    correlation_matrix['gross']
    .drop('gross')
    .sort_values(key=lambda values: values.abs(), ascending=False)
)

gross_correlations

## 9. Encode categorical fields carefully

Some tutorials convert text columns into category codes to include them in a correlation matrix. That can be useful for exploration, but it has a limitation: category codes are arbitrary labels, not true numeric measurements.

We use a copy called `df_numerized` so the original `df` stays intact.

In [ ]:
# Copy before encoding
df_numerized = df.copy()

# Encode text columns
for col_name in df_numerized.columns:
    if df_numerized[col_name].dtype == 'object':
        df_numerized[col_name] = df_numerized[col_name].astype('category')
        df_numerized[col_name] = df_numerized[col_name].cat.codes

df_numerized.head()

In [ ]:
# Full encoded correlation
encoded_correlation_matrix = df_numerized.corr(method='pearson')

plt.figure(figsize=(14, 10))

sns.heatmap(
    encoded_correlation_matrix,
    annot=True,
    cmap='coolwarm',
    linewidths=0.5,
    fmt='.2f'
)

plt.title('Correlation Matrix with Encoded Categories')
plt.xlabel('Movie Features')
plt.ylabel('Movie Features')
plt.show()

## 10. Top correlations after encoding

This view helps identify which columns are most associated with `gross`, including encoded categorical fields. Treat encoded categorical correlations as signals for follow-up, not as final proof.

In [ ]:
# Rank encoded correlations
gross_encoded_correlations = (
    encoded_correlation_matrix['gross']
    .drop('gross')
    .sort_values(key=lambda values: values.abs(), ascending=False)
)

gross_encoded_correlations

## 11. Top companies by gross revenue

Now that `df` still contains `company`, we can group by company correctly. This is where the earlier error happened: if `df` is overwritten with numeric columns only, `company` disappears.

In [ ]:
# Sum gross by company
company_gross_sum = df.groupby('company', dropna=False)[['gross']].sum()

# Keep top companies
company_gross_sum_sorted = company_gross_sum.sort_values(
    by='gross',
    ascending=False
).head(15)

# Format as integers
company_gross_sum_sorted['gross'] = company_gross_sum_sorted['gross'].astype('int64')

company_gross_sum_sorted

In [ ]:
# Plot top companies
plt.figure(figsize=(12, 6))

sns.barplot(
    x='gross',
    y=company_gross_sum_sorted.index,
    data=company_gross_sum_sorted,
    hue=company_gross_sum_sorted.index,
    palette='Blues_r',
    legend=False
)

plt.title('Top 15 Companies by Gross Revenue')
plt.xlabel('Total Gross Revenue')
plt.ylabel('Company')
plt.show()

## 12. Takeaways

- `budget` and `votes` are expected to be strong candidates for explaining gross revenue.
- `score` can have a relationship with revenue, but it is not enough by itself.
- Company-level totals show which studios dominate total gross revenue in this dataset.
- Encoded categorical correlations can suggest patterns, but they should be interpreted carefully.
- The main technical fix was preserving `df` and using separate working dataframes.